# DP1 Inner-Bin Appendix Stamp Grid

Run this notebook inside the Rubin Science Platform. It selects 12 representative inner-bin foreground-background pairs and renders a 3x4 grid of r-band deep-coadd postage stamps for appendix validation.

The selection is designed to span separation, catalog deblend overlap, foreground/background r-band flux ratio, and foreground/background color while avoiding extremely one-sided flux ratios where one member would be hard to see.

In [ ]:
from pathlib import Path
import io
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
from astropy.wcs import WCS
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src" / "dusty_colors").exists():
    ROOT = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(ROOT / "src"))

from dusty_colors.color_split_bias import (
    attach_independent_proxy_columns,
    build_radial_pair_table,
    radial_bin_label,
)
from dusty_colors.postage_stamps import (
    available_tap_columns,
    fetch_object_metadata,
    requested_blend_columns,
    select_existing_columns,
)

STACK_ID = "dp1_default"
BIN_INDEX = 0
COLOR = "g-z"
BAND = "r"
STAMP_FOV_ARCSEC = 8.0
GRID_SHAPE = (3, 4)
N_STAMPS = GRID_SHAPE[0] * GRID_SHAPE[1]
FLUX_RATIO_LIMIT_DEX = 1.0
REFETCH_CUTOUTS = False
CUTOUT_CACHE_DIRNAME = f"appendix_{BAND}_band_stamp_grid_cutouts"
# Optional control for removing visually awkward examples from the grid.
# Add one or more pair IDs, e.g. {"dp1_default_bin1_0057"}.
EXCLUDED_PAIR_IDS = {
    "dp1_default_bin1_0057",
    "dp1_default_bin1_0041",
    "dp1_default_bin1_0031",
    "dp1_default_bin1_0032",
    "dp1_default_bin1_0053",
    "dp1_default_bin1_0056",
    "dp1_default_bin1_0019",
}

CONFIG_PATH = ROOT / "results" / "stacks" / STACK_ID / "config_resolved.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = ROOT / "configs" / "analyses" / f"{STACK_ID}.yaml"
SAMPLE_DIR = ROOT / "results" / "samples" / STACK_ID
OUT_DIR = ROOT / "results" / "postage_stamps" / STACK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
CUTOUT_CACHE_DIR = OUT_DIR / CUTOUT_CACHE_DIRNAME

print(f"stack: {STACK_ID}")
print(f"config: {CONFIG_PATH}")
print(f"outputs: {OUT_DIR}")

In [ ]:
try:
    from lsst.rsp import get_tap_service
    from lsst.rsp.service import get_siav2_service
    from lsst.rsp.utils import get_pyvo_auth
    from pyvo.dal.adhoc import DatalinkResults, SodaQuery
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs to run inside the Rubin Science Platform Notebook Aspect."
    ) from exc

tap_service = get_tap_service("tap")
sia_service = get_siav2_service("dp1")
session = get_pyvo_auth()

In [ ]:
with open(CONFIG_PATH) as handle:
    config = yaml.safe_load(handle)

analysis_config = config.get(
    "stack", config.get("analysis", {}).get("data", {}).get("stack", {})
)
r_edges = np.asarray(analysis_config["r_bin_edges"], dtype=float)
r_min, r_max = float(r_edges[BIN_INDEX]), float(r_edges[BIN_INDEX + 1])
bin_label = radial_bin_label(BIN_INDEX, r_min, r_max)

foreground = pd.read_parquet(SAMPLE_DIR / "foreground.parquet")
background = pd.read_parquet(SAMPLE_DIR / "background.parquet")
pairs = build_radial_pair_table(
    foreground,
    background,
    stack_id=STACK_ID,
    bin_index=BIN_INDEX,
    r_edges_kpc=r_edges,
)
print(f"{bin_label}: {len(pairs)} pairs")
display(pairs.head())

In [ ]:
metadata_path = OUT_DIR / "object_blend_metadata.csv"
object_ids = sorted(
    set(pairs["foreground_object_id"].dropna().astype(int))
    | set(pairs["background_object_id"].dropna().astype(int))
)

if metadata_path.exists():
    object_metadata = pd.read_csv(metadata_path)
else:
    object_metadata = pd.DataFrame()

have_ids = set()
if not object_metadata.empty:
    id_col = "objectId" if "objectId" in object_metadata.columns else "object_id"
    have_ids = set(object_metadata[id_col].dropna().astype(int))

missing_ids = sorted(set(object_ids) - have_ids)
if missing_ids:
    available = available_tap_columns(tap_service, "dp1.Object")
    requested = requested_blend_columns(bands=("g", "r", "i", "z"))
    columns = select_existing_columns(requested, available)
    fetched = fetch_object_metadata(
        tap_service, missing_ids, columns, table_name="dp1.Object"
    )
    object_metadata = pd.concat([object_metadata, fetched], ignore_index=True)
    id_col = "objectId" if "objectId" in object_metadata.columns else "object_id"
    object_metadata = object_metadata.drop_duplicates(subset=[id_col], keep="last")
    object_metadata.to_csv(metadata_path, index=False)

print(f"Object metadata rows: {len(object_metadata)}")
display(object_metadata.head())

In [ ]:
prepared = attach_independent_proxy_columns(
    pairs,
    foreground,
    background,
    color=COLOR,
    snr_max=analysis_config.get("snr_max", 100.0),
    object_metadata=object_metadata,
    overlap_bands=("r", "i"),
)

fg_rows = foreground.iloc[prepared["foreground_index"].to_numpy(int)].reset_index(
    drop=True
)
bg_rows = background.iloc[prepared["background_index"].to_numpy(int)].reset_index(
    drop=True
)

with np.errstate(divide="ignore", invalid="ignore"):
    prepared["foreground_r_flux"] = fg_rows["flux_r"].to_numpy(float)
    prepared["background_r_flux"] = bg_rows["flux_r"].to_numpy(float)
    prepared["foreground_background_r_flux_ratio"] = (
        prepared["foreground_r_flux"] / prepared["background_r_flux"]
    )
    prepared["log10_foreground_background_r_flux_ratio"] = np.log10(
        prepared["foreground_background_r_flux_ratio"]
    )

summary_cols = [
    "pair_id",
    "theta_arcsec",
    "r_perp_kpc",
    "max_deblend_fluxOverlapFraction",
    "log10_foreground_background_r_flux_ratio",
    "foreground_g_z",
    "background_g_z",
]
display(prepared[summary_cols].describe(include="all"))

In [ ]:
def _finite_metric_pool(df, metric):
    values = pd.to_numeric(df[metric], errors="coerce")
    return df.loc[np.isfinite(values)].copy()


def _choose_nearest(
    df, metric, target, selected_ids, tag, *, restrict_visibility=False
):
    pool = _finite_metric_pool(df, metric)
    if restrict_visibility and "log10_foreground_background_r_flux_ratio" in pool:
        ratio = pd.to_numeric(
            pool["log10_foreground_background_r_flux_ratio"], errors="coerce"
        )
        pool = pool.loc[np.isfinite(ratio) & (np.abs(ratio) <= FLUX_RATIO_LIMIT_DEX)]
    pool = pool.loc[~pool["pair_id"].isin(selected_ids)]
    if pool.empty:
        return None
    values = pd.to_numeric(pool[metric], errors="coerce")
    index = (values - target).abs().idxmin()
    row = pool.loc[index].copy()
    row["selection_tag"] = tag
    return row


def _choose_extreme(
    df, metric, selected_ids, tag, *, high=True, restrict_visibility=False
):
    pool = _finite_metric_pool(df, metric)
    if restrict_visibility and "log10_foreground_background_r_flux_ratio" in pool:
        ratio = pd.to_numeric(
            pool["log10_foreground_background_r_flux_ratio"], errors="coerce"
        )
        pool = pool.loc[np.isfinite(ratio) & (np.abs(ratio) <= FLUX_RATIO_LIMIT_DEX)]
    pool = pool.loc[~pool["pair_id"].isin(selected_ids)]
    if pool.empty:
        return None
    values = pd.to_numeric(pool[metric], errors="coerce")
    index = values.idxmax() if high else values.idxmin()
    row = pool.loc[index].copy()
    row["selection_tag"] = tag
    return row


def select_representative_pairs(df, n=N_STAMPS, excluded_pair_ids=None):
    excluded_pair_ids = set(excluded_pair_ids or [])
    valid = df.loc[~df["pair_id"].isin(excluded_pair_ids)].copy()
    if excluded_pair_ids:
        print(f"Excluded from stamp grid: {sorted(excluded_pair_ids)}")
    selected = []
    selected_ids = set()

    def add(row):
        if row is None:
            return
        pid = row["pair_id"]
        if pid not in selected_ids:
            selected.append(row)
            selected_ids.add(pid)

    metric_specs = [
        ("theta_arcsec", 0.05, "small separation", False),
        ("theta_arcsec", 0.95, "large separation", False),
        ("max_deblend_fluxOverlapFraction", 0.10, "low blendedness", False),
        ("max_deblend_fluxOverlapFraction", 0.50, "typical blendedness", False),
        ("max_deblend_fluxOverlapFraction", 0.90, "high blendedness", False),
        ("log10_foreground_background_r_flux_ratio", 0.25, "fg faint in r", True),
        ("log10_foreground_background_r_flux_ratio", 0.75, "fg bright in r", True),
        ("foreground_g_z", 0.15, "blue foreground", True),
        ("foreground_g_z", 0.85, "red foreground", True),
        ("background_g_z", 0.15, "blue background", True),
        ("background_g_z", 0.85, "red background", True),
    ]

    for metric, quantile, tag, restrict_visibility in metric_specs:
        pool = _finite_metric_pool(valid, metric)
        if pool.empty:
            continue
        target = float(
            np.nanquantile(pd.to_numeric(pool[metric], errors="coerce"), quantile)
        )
        add(
            _choose_nearest(
                valid,
                metric,
                target,
                selected_ids,
                tag,
                restrict_visibility=restrict_visibility,
            )
        )

    # Fill any remaining slots with pairs that are typical in the available metrics.
    metrics = [
        "theta_arcsec",
        "max_deblend_fluxOverlapFraction",
        "log10_foreground_background_r_flux_ratio",
        "foreground_g_z",
        "background_g_z",
    ]
    pool = valid.loc[~valid["pair_id"].isin(selected_ids)].copy()
    if not pool.empty:
        score = np.zeros(len(pool), dtype=float)
        used = 0
        for metric in metrics:
            values = pd.to_numeric(pool[metric], errors="coerce").to_numpy(float)
            finite = np.isfinite(values)
            if not finite.any():
                continue
            med = np.nanmedian(values)
            scale = np.nanpercentile(values, 84) - np.nanpercentile(values, 16)
            if not np.isfinite(scale) or scale == 0:
                scale = np.nanstd(values)
            if not np.isfinite(scale) or scale == 0:
                continue
            metric_score = np.abs((values - med) / scale)
            metric_score[~finite] = np.nanmedian(metric_score[finite])
            score += metric_score
            used += 1
        pool["typicality_score"] = score / max(used, 1)
        ratio = pd.to_numeric(
            pool["log10_foreground_background_r_flux_ratio"], errors="coerce"
        )
        visible_pool = pool.loc[
            np.isfinite(ratio) & (np.abs(ratio) <= FLUX_RATIO_LIMIT_DEX)
        ]
        if not visible_pool.empty:
            pool = visible_pool
        for _, row in pool.sort_values("typicality_score").iterrows():
            row = row.copy()
            row["selection_tag"] = "typical"
            add(row)
            if len(selected) >= n:
                break

    return pd.DataFrame(selected).head(n).reset_index(drop=True)


selected_pairs = select_representative_pairs(
    prepared, N_STAMPS, excluded_pair_ids=EXCLUDED_PAIR_IDS
)
selected_path = OUT_DIR / "appendix_r_band_stamp_grid_pairs.csv"
selected_pairs.to_csv(selected_path, index=False)
display(selected_pairs[summary_cols + ["selection_tag"]])
print(f"Wrote {selected_path}")

In [ ]:
def find_deep_coadd_access_url(ra, dec, band, search_radius_deg=0.001):
    # PyVO SIA2 expects POS as a CIRCLE tuple: (ra, dec, radius), in degrees.
    # For Rubin DP1 deep coadds, restrict the SIA result set with calib_level
    # and dpsubtype, then filter the returned metadata table by lsst_band.
    results = sia_service.search(
        pos=(float(ra), float(dec), float(search_radius_deg)),
        calib_level=3,
        dpsubtype="lsst.deep_coadd",
    )
    table = results.to_table().to_pandas()
    if table.empty:
        raise RuntimeError(f"No SIA image found for band={band} at ra={ra}, dec={dec}")

    if "lsst_band" in table.columns:
        table = table.loc[table["lsst_band"].astype(str) == str(band)]
    if "dataproduct_subtype" in table.columns:
        table = table.loc[table["dataproduct_subtype"].astype(str) == "lsst.deep_coadd"]
    if table.empty or "access_url" not in table.columns:
        raise RuntimeError(
            f"No deep-coadd access_url found for band={band} at ra={ra}, dec={dec}"
        )
    return str(table.iloc[0]["access_url"])


def get_cutout_hdul(access_url, ra, dec, fov_arcsec):
    datalink = DatalinkResults.from_result_url(access_url, session=session)
    cutout_service = datalink.get_adhocservice_by_id("cutout-sync")
    query = SodaQuery.from_resource(datalink, cutout_service, session=session)
    query.circle = (
        float(ra) * u.deg,
        float(dec) * u.deg,
        (0.5 * float(fov_arcsec) / 3600.0) * u.deg,
    )
    response = query.execute_stream()
    payload = response.read()
    query.raise_if_error()
    return fits.open(io.BytesIO(payload))


def science_hdu(hdul):
    for hdu in hdul:
        data = getattr(hdu, "data", None)
        if data is not None and np.ndim(data) == 2:
            return hdu
    raise RuntimeError("No 2D science image found in cutout FITS")


def safe_filename_fragment(value):
    return "".join(
        char if char.isalnum() or char in {"-", "_"} else "_" for char in str(value)
    )


def cutout_cache_path(panel_index, pair_id):
    filename = f"{int(panel_index):02d}_{safe_filename_fragment(pair_id)}_{BAND}.fits"
    return CUTOUT_CACHE_DIR / filename

In [ ]:
CUTOUT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
cutout_status_path = OUT_DIR / f"appendix_{BAND}_band_stamp_grid_cutout_status.csv"

fetch_statuses = []
for panel_index, (_, row) in enumerate(selected_pairs.iterrows(), start=1):
    cache_path = cutout_cache_path(panel_index, row["pair_id"])
    try:
        if cache_path.exists() and not REFETCH_CUTOUTS:
            fetch_statuses.append(
                {
                    "panel_index": panel_index,
                    "pair_id": row["pair_id"],
                    "status": "cached",
                    "cutout_path": str(cache_path),
                    "message": "",
                }
            )
            continue

        access_url = find_deep_coadd_access_url(
            row["midpoint_ra"], row["midpoint_dec"], BAND
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            hdul = get_cutout_hdul(
                access_url,
                row["midpoint_ra"],
                row["midpoint_dec"],
                STAMP_FOV_ARCSEC,
            )
        hdul.writeto(cache_path, overwrite=True)
        hdul.close()
        fetch_statuses.append(
            {
                "panel_index": panel_index,
                "pair_id": row["pair_id"],
                "status": "fetched",
                "cutout_path": str(cache_path),
                "message": "",
            }
        )
    except Exception as exc:
        fetch_statuses.append(
            {
                "panel_index": panel_index,
                "pair_id": row["pair_id"],
                "status": "failed",
                "cutout_path": str(cache_path),
                "message": str(exc),
            }
        )

cutout_fetch_status = pd.DataFrame(fetch_statuses)
cutout_fetch_status.to_csv(cutout_status_path, index=False)
display(cutout_fetch_status)
print(f"Wrote {cutout_status_path}")
print(f"Cached cutouts in {CUTOUT_CACHE_DIR}")

## Plot Cached Cutouts

After the cutout-cache cell has run once, iterate on figure aesthetics by rerunning only the next cell. It reads the cached FITS files and does not contact the Rubin image services.


In [ ]:
FIGURE_SIZE = (7, 5)
FIGURE_DPI = 500
STAMP_INTERVAL_PERCENTILE = 99.0
PANEL_TITLE_FONTSIZE = 8
FOREGROUND_MARKER_SIZE = 75
BACKGROUND_MARKER_SIZE = 95
CONNECTOR_LINEWIDTH = 1.0

fig = plt.figure(figsize=FIGURE_SIZE, constrained_layout=True)
plot_statuses = []

panel_indices = [5, 6, 2, 3, 4, 7, 8, 9, 10, 11, 12, 1]
for cache_index, (_, row) in enumerate(selected_pairs.iterrows(), start=1):
    cache_path = cutout_cache_path(cache_index, row["pair_id"])
    try:
        if not cache_path.exists():
            raise RuntimeError(
                f"Missing cached cutout: {cache_path}. Run the cutout-cache cell above first."
            )
        with fits.open(cache_path) as hdul:
            hdu = science_hdu(hdul)
            data = np.asarray(hdu.data, dtype=float).copy()
            header = hdu.header.copy()
        wcs = WCS(header)

        panel_index = panel_indices[cache_index - 1]
        ax = fig.add_subplot(GRID_SHAPE[0], GRID_SHAPE[1], panel_index, projection=wcs)
        interval = PercentileInterval(STAMP_INTERVAL_PERCENTILE)
        norm = ImageNormalize(data, interval=interval, stretch=AsinhStretch())
        ax.imshow(data, origin="lower", cmap="gray", norm=norm)
        world = ax.get_transform("world")
        ax.scatter(
            row["foreground_ra"],
            row["foreground_dec"],
            transform=world,
            marker="o",
            facecolors="none",
            edgecolors="tab:cyan",
            s=FOREGROUND_MARKER_SIZE,
            linewidths=1.5,
        )
        ax.scatter(
            row["background_ra"],
            row["background_dec"],
            transform=world,
            marker="+",
            color="tab:red",
            s=BACKGROUND_MARKER_SIZE,
            linewidths=1.8,
        )
        ax.plot(
            [row["foreground_ra"], row["background_ra"]],
            [row["foreground_dec"], row["background_dec"]],
            transform=world,
            color="gold",
            linewidth=CONNECTOR_LINEWIDTH,
        )
        overlap = row.get("max_deblend_fluxOverlapFraction", np.nan)
        ratio = row.get("foreground_background_r_flux_ratio", np.nan)
        ax.set_title(f"{row['selection_tag']}")
        ax.coords[0].set_axislabel("")
        ax.coords[1].set_axislabel("")
        ax.coords[0].set_ticklabel_visible(False)
        ax.coords[1].set_ticklabel_visible(False)
        ax.coords[0].set_ticks_visible(False)
        ax.coords[1].set_ticks_visible(False)
        plot_statuses.append({"pair_id": row["pair_id"], "status": "ok", "message": ""})
    except Exception as exc:
        ax = fig.add_subplot(GRID_SHAPE[0], GRID_SHAPE[1], panel_index)
        ax.text(
            0.5, 0.5, f"{row['pair_id']}\n{exc}", ha="center", va="center", fontsize=8
        )
        ax.set_axis_off()
        plot_statuses.append(
            {"pair_id": row["pair_id"], "status": "failed", "message": str(exc)}
        )

png_path = OUT_DIR / f"appendix_{BAND}_band_stamp_grid.png"
pdf_path = OUT_DIR / f"appendix_{BAND}_band_stamp_grid.pdf"
status_path = OUT_DIR / f"appendix_{BAND}_band_stamp_grid_status.csv"
fig.savefig(png_path, dpi=FIGURE_DPI)
fig.savefig(pdf_path)
pd.DataFrame(plot_statuses).to_csv(status_path, index=False)
plt.show()

print(f"Wrote {png_path}")
print(f"Wrote {pdf_path}")
print(f"Wrote {status_path}")

## Figure Notes

Blue circles mark foreground objects, red plus signs mark background objects, and the gold segment connects the pair.